# FlowGuard - Setup & Validation
Mount Drive, install deps, validate raw data.

In [ ]:
import os

# For Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = "/content/drive/MyDrive/flowguard"
    if not os.path.exists(REPO_DIR):
        print("Clone the repo to Google Drive first, or adjust REPO_DIR.")
    else:
        os.chdir(REPO_DIR)
except ImportError:
    # Local development
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q flwr[simulation] shap umap-learn advertorch wandb pyarrow pyyaml tqdm plotly
!pip install -e . -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
from src.data.validate_raw import validate_raw_datasets

report = validate_raw_datasets(config_path="configs/base.yaml")
all_ok = all(info["exists"] for info in report.values())

if not all_ok:
    print("\nMISSING DATASETS:")
    for ds_name, info in report.items():
        if not info["exists"]:
            print(f"  -> {info['expected_file']}")
    print("\nDownload from: https://staff.itee.uq.edu.au/marius/NIDS_datasets/")
else:
    print("\nAll datasets found!")
    for ds_name, info in report.items():
        print(f"  {ds_name}: {info['rows']:,} rows, {info['cols']} cols, {info['size_mb']:.1f} MB")